#INFORMACION DE LA TABLA

In [0]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(tracking_id) AS tracking_id_no_nulo,
    COUNT(pedido_id) AS pedido_id_no_nulo,
    COUNT(courier) AS courier_no_nulo,
    COUNT(estado_entrega) AS estado_no_nulo,
    COUNT(sucursal_origen) AS sucursal_no_nulo,
    COUNT(fecha_actualizacion) AS fecha_no_nula
FROM trackingenvios_catalog.dbo.trackingenvios

In [0]:
%sql
SELECT
    estado_entrega,
    COUNT(*) AS cantidad
FROM trackingenvios_catalog.dbo.trackingenvios
GROUP BY estado_entrega
ORDER BY cantidad DESC;

#---------------------------------------------------------------------------------
#CREACION DE ESQUEMAS

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electrocasa.bronze;

CREATE SCHEMA IF NOT EXISTS electrocasa.silver;

CREATE SCHEMA IF NOT EXISTS electrocasa.gold;

#---------------------------------------------------------------------------------
#CREACION DE CATALOG

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS electrocasa;

#---------------------------------------------------------------------------------
#CREACION DE VOLUME
###Aqui es donde se alojaran los archivos necesarios

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS electrocasa.bronze.landing;

In [0]:
from pathlib import Path

base = Path("/Volumes/electrocasa/bronze/landing")

carpetas = [
    "ventas",
    "catalogo",
    "empleados",
    "resenas",
    "devoluciones"
]

for carpeta in carpetas:
    (base / carpeta).mkdir(parents=True, exist_ok=True)

print("Carpetas creadas correctamente.")

#---------------------------------------------------------------------------------
#GOLD

#----------------------------------------------------------------------------
#GRUPOS

###los grupos fueron creados manualmente desde settings

##ingenieria

In [0]:
%sql
GRANT USE SCHEMA
ON SCHEMA electrocasa.bronze
TO `Ingenieria`;

GRANT USE SCHEMA
ON SCHEMA electrocasa.silver
TO `Ingenieria`;

GRANT USE SCHEMA
ON SCHEMA electrocasa.gold
TO `Ingenieria`;

In [0]:
%sql
GRANT SELECT, MODIFY
ON SCHEMA electrocasa.bronze
TO `Ingenieria`;

GRANT SELECT, MODIFY
ON SCHEMA electrocasa.silver
TO `Ingenieria`;

GRANT SELECT, MODIFY
ON SCHEMA electrocasa.gold
TO `Ingenieria`;

##analistas

In [0]:
%sql
GRANT USE CATALOG
ON CATALOG electrocasa
TO `Analistas`;

In [0]:
%sql
GRANT USE SCHEMA
ON SCHEMA electrocasa.gold
TO `Analistas`;

In [0]:
%sql
GRANT SELECT
ON SCHEMA electrocasa.gold
TO `Analistas`;

##auditoria

In [0]:
%sql
GRANT USE CATALOG
ON CATALOG electrocasa
TO `Auditoria`;

In [0]:
%sql
GRANT USE SCHEMA
ON SCHEMA electrocasa.gold
TO `Auditoria`;

In [0]:
%sql
GRANT SELECT
ON SCHEMA electrocasa.gold
TO `Auditoria`;

#MASKING

In [0]:
%sql
DESCRIBE FUNCTION mask;

In [0]:
%sql
SHOW FUNCTIONS LIKE '*FILTER*';

In [0]:
%sql
CREATE OR REPLACE FUNCTION electrocasa.silver.mask_dni(valor STRING)
RETURNS STRING
RETURN CASE
    WHEN is_account_group_member('Ingenieria') THEN valor
    ELSE mask(valor)
END;

In [0]:
%sql
CREATE OR REPLACE VIEW electrocasa.gold.empleados_activos_seguro AS
SELECT
    id_empleado,
    nombre,

    CASE
        WHEN is_account_group_member('Ingenieria') THEN dni
        ELSE mask(dni)
    END AS dni,

    email,

    CASE
        WHEN is_account_group_member('Ingenieria')
            THEN CAST(salario AS STRING)
        ELSE mask(CAST(salario AS STRING))
    END AS salario,

    sucursal_id,
    cargo,
    tipo_evento,
    fecha_evento,
    fecha_inicio,
    fecha_fin,
    es_actual

FROM electrocasa.silver.empleados_silver;

In [0]:
%sql
GRANT SELECT
ON VIEW electrocasa.gold.empleados_activos_seguro
TO `Analistas`;

GRANT SELECT
ON VIEW electrocasa.gold.empleados_activos_seguro
TO `Auditoria`;